# StreamLoop: ajuste del modelo de churn

El objetivo es detectar clientes que cancelarán. Como los falsos negativos son especialmente costosos, la búsqueda prioriza F2 (beta=2), que da más peso al recall, y también considera la estabilidad entre folds.

El test se reserva para dos evaluaciones comparables: la línea base y el modelo final. Instala `pandas`, `scipy` y `scikit-learn` si aún no están disponibles en el entorno.

In [8]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, fbeta_score, make_scorer,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_URL = (
    'https://raw.githubusercontent.com/IBM/'
    'telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
)
RANDOM_STATE = 42
F2_SCORER = make_scorer(fbeta_score, beta=2, pos_label=1, zero_division=0)

## 1. Carga, revisión y limpieza mínima

Se convierte `TotalCharges` a numérico para que los valores vacíos pasen a ser faltantes imputables. `customerID` se elimina por ser un identificador, no una señal útil para generalizar.

In [9]:
data = pd.read_csv(DATA_URL)
initial_rows = len(data)
data = data.drop_duplicates().copy()
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
missing_by_column = data.isna().sum()
missing_by_column = missing_by_column[missing_by_column > 0].to_dict()
target = data.pop('Churn').map({'No': 0, 'Yes': 1})
if target.isna().any():
    raise ValueError('Churn contiene etiquetas desconocidas o vacías.')
features = data.drop(columns=['customerID'], errors='ignore')
numeric_columns = features.select_dtypes(include=np.number).columns.tolist()
categorical_columns = features.select_dtypes(exclude=np.number).columns.tolist()

print(f'Filas originales: {initial_rows}; tras quitar duplicados: {len(data)}')
print(f'Predictores: {features.shape[1]}')
print('Tipos de datos:\n', features.dtypes.value_counts())
print('Columnas con valores faltantes:', missing_by_column)
print(f'Proporción de churn: {target.mean():.1%}')

Filas originales: 7043; tras quitar duplicados: 7043
Predictores: 19
Tipos de datos:
 str        15
int64       2
float64     2
Name: count, dtype: int64
Columnas con valores faltantes: {'TotalCharges': 11}
Proporción de churn: 26.5%


## 2. División y pipeline

La separación estratificada ocurre antes de ajustar transformaciones. Imputación, escalado y one-hot encoding se aprenden dentro del `Pipeline` en cada fold de entrenamiento.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=RANDOM_STATE, stratify=target
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_columns),
    ('categorical', categorical_pipeline, categorical_columns),
])
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression()),
])
print(f'Entrenamiento: {len(X_train)} filas; test reservado: {len(X_test)} filas')

Entrenamiento: 5634 filas; test reservado: 1409 filas


## 3. Línea base

Se conserva la regresión logística con hiperparámetros por defecto. Esta es la primera y única evaluación de la línea base en test.

In [11]:
def evaluate_once(model, X, y):
    probabilities = model.predict_proba(X)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(y, predictions),
        'precision': precision_score(y, predictions, zero_division=0),
        'recall': recall_score(y, predictions, zero_division=0),
        'f1': f1_score(y, predictions, zero_division=0),
        'f2': fbeta_score(y, predictions, beta=2, zero_division=0),
        'roc_auc': roc_auc_score(y, probabilities),
        'confusion_matrix': confusion_matrix(y, predictions, labels=[0, 1]),
    }

baseline = pipeline.fit(X_train, y_train)
baseline_metrics = evaluate_once(baseline, X_test, y_test)
pd.Series({k: v for k, v in baseline_metrics.items() if k != 'confusion_matrix'}, name='Línea base')

accuracy     0.805536
precision    0.657233
recall       0.558824
f1           0.604046
f2           0.576075
roc_auc      0.841861
Name: Línea base, dtype: float64

## 4. Búsqueda amplia con RandomizedSearchCV

F2 favorece recall frente a precisión, en línea con el coste de no detectar cancelaciones. La validación cruzada y toda búsqueda usan únicamente el conjunto de entrenamiento.

In [12]:
randomized_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions={
        'classifier__C': loguniform(0.01, 100),
        'classifier__class_weight': [None, 'balanced'],
    },
    n_iter=16, scoring=F2_SCORER, cv=cv, refit=True, n_jobs=-1,
    random_state=RANDOM_STATE, return_train_score=False,
)
randomized_search.fit(X_train, y_train)
random_results = pd.DataFrame(randomized_search.cv_results_)
random_results[['params', 'mean_test_score', 'std_test_score']]\
    .sort_values('mean_test_score', ascending=False).head(5)

,params,mean_test_score,std_test_score
5,"{'classifier__C': 0.21618942406574443, 'classi...",0.721570,0.028751
6,"{'classifier__C': 6.79657809075816, 'classifie...",0.721305,0.030422
8,"{'classifier__C': 21.368329072358772, 'classif...",0.720437,0.030462
14,"{'classifier__C': 0.03613894271216528, 'classi...",0.719954,0.025338
1,"{'classifier__C': 0.05416754583247458, 'classi...",0.719703,0.025757


## 5. Refinamiento con GridSearchCV

Se inspeccionan los resultados aleatorios y se refina `C` alrededor de los tres mejores candidatos. Para seleccionar el estimador reajustado se maximiza `F2 medio - 0.5 × desviación estándar`, evitando decidir solo por el promedio.

In [13]:
top_random = random_results.sort_values(
    ['mean_test_score', 'std_test_score'], ascending=[False, True]
).head(3)
c_values = sorted({
    round(float(value) * factor, 8)
    for value in top_random['param_classifier__C']
    for factor in (0.5, 1.0, 2.0)
})
class_weights = sorted({str(value) for value in top_random['param_classifier__class_weight']})
weight_values = [None if value == 'None' else value for value in class_weights]

def select_stable_candidate(cv_results):
    adjusted = cv_results['mean_test_score'] - 0.5 * cv_results['std_test_score']
    return int(np.nanargmax(adjusted))

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid={'classifier__C': c_values, 'classifier__class_weight': weight_values},
    scoring=F2_SCORER, cv=cv, refit=select_stable_candidate, n_jobs=-1,
    return_train_score=False,
)
grid_search.fit(X_train, y_train)
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results[['params', 'mean_test_score', 'std_test_score']]\
    .sort_values('mean_test_score', ascending=False).head(5)

,params,mean_test_score,std_test_score
3,"{'classifier__C': 3.39828905, 'classifier__cla...",0.722760,0.029922
2,"{'classifier__C': 0.43237885, 'classifier__cla...",0.721877,0.028751
1,"{'classifier__C': 0.21618942, 'classifier__cla...",0.721570,0.028751
4,"{'classifier__C': 6.79657809, 'classifier__cla...",0.721305,0.030422
0,"{'classifier__C': 0.10809471, 'classifier__cla...",0.720725,0.026746


## 6. Evaluación final y reporte

`GridSearchCV` ya reajustó el mejor estimador (`refit`); no se vuelve a entrenar manualmente. Esta es la segunda y última evaluación en test. Un mayor recall puede implicar menor precisión y más falsos positivos, por lo que se comparan ambas métricas y las matrices de confusión.

In [14]:
final_metrics = evaluate_once(grid_search.best_estimator_, X_test, y_test)
selected_cv = grid_results.iloc[grid_search.best_index_]
randomized_best = random_results.iloc[randomized_search.best_index_]
metric_names = ['accuracy', 'precision', 'recall', 'f1', 'f2', 'roc_auc']
comparison = pd.DataFrame({
    'Línea base': {name: baseline_metrics[name] for name in metric_names},
    'Modelo final': {name: final_metrics[name] for name in metric_names},
})
display(comparison)
print('Matriz baseline [[TN, FP], [FN, TP]]:\n', baseline_metrics['confusion_matrix'])
print('Matriz final [[TN, FP], [FN, TP]]:\n', final_metrics['confusion_matrix'])
print('Hiperparámetros elegidos:', grid_search.best_params_)
print(f'F2 CV elegido: {selected_cv.mean_test_score:.4f} ± {selected_cv.std_test_score:.4f}')
print(f'Cambio de recall: {final_metrics["recall"] - baseline_metrics["recall"]:+.4f}')
print(f'Cambio de precisión: {final_metrics["precision"] - baseline_metrics["precision"]:+.4f}')

def candidate_table(results, count=5):
    top = results.sort_values(['mean_test_score', 'std_test_score'], ascending=[False, True]).head(count)
    lines = ['| Puesto | F2 medio | Desv. estándar | Hiperparámetros |', '| ---: | ---: | ---: | --- |']
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        params = json.dumps(row['params'], ensure_ascii=True, sort_keys=True)
        lines.append(f"| {rank} | {row['mean_test_score']:.4f} | {row['std_test_score']:.4f} | `{params}` |")
    return '\n'.join(lines)

metric_rows = '\n'.join(
    f"| {name} | {baseline_metrics[name]:.4f} | {final_metrics[name]:.4f} |"
    for name in metric_names
)
report = f"""# StreamLoop: informe de ajuste del modelo de churn

## Datos y preparación

- Fuente: [dataset Telco Customer Churn de IBM]({DATA_URL}).
- Filas originales: {initial_rows}; tras quitar duplicados: {len(data)}; predictores: {features.shape[1]} (se excluyó `customerID`).
- Entrenamiento: {len(X_train)} filas; test: {len(X_test)} filas; proporción churn en test: {y_test.mean():.1%}.
- Faltantes antes del pipeline: `{json.dumps(missing_by_column, sort_keys=True)}`. `TotalCharges` se convierte a numérico; imputación, escalado y codificación se ajustan dentro del pipeline.
- División estratificada 80/20 (`random_state={RANDOM_STATE}`); validación cruzada estratificada de 5 folds.

## Métrica y método

La métrica de búsqueda es F2 para `Churn=Yes` (`beta=2`), que prioriza recall por el coste de no detectar cancelaciones. El candidato final maximiza `F2 medio - 0.5 × desviación estándar entre folds`.

- RandomizedSearchCV: 16 candidatos, `C ~ loguniform(0.01, 100)`, `class_weight` en `{None, 'balanced'}`, `n_jobs=-1`.
- GridSearchCV: refinamiento de `C` alrededor de los tres mejores candidatos aleatorios y pesos observados; `n_jobs=-1`.
- Mejor candidato aleatorio: `{json.dumps(randomized_search.best_params_, sort_keys=True)}`; F2 CV `{randomized_best.mean_test_score:.4f} ± {randomized_best.std_test_score:.4f}`.

### Candidatos aleatorios

{candidate_table(random_results)}

### Candidatos de grilla

{candidate_table(grid_results)}

## Comparación en test

Test evaluado exactamente dos veces: baseline y modelo final. Búsquedas ajustadas únicamente sobre entrenamiento. Métricas adicionales a F2: precisión, recall, F1, accuracy, ROC-AUC y matriz de confusión.

| Métrica | Línea base | Modelo final |
| --- | ---: | ---: |
{metric_rows}

| Matriz `[[TN, FP], [FN, TP]]` | `{baseline_metrics['confusion_matrix'].tolist()}` | `{final_metrics['confusion_matrix'].tolist()}` |

## Selección y trade-off

- Hiperparámetros finales: `{json.dumps(grid_search.best_params_, sort_keys=True)}`.
- F2 CV seleccionado: `{selected_cv.mean_test_score:.4f} ± {selected_cv.std_test_score:.4f}`; criterio de estabilidad: `{selected_cv.mean_test_score - 0.5 * selected_cv.std_test_score:.4f}`.
- Cambio de recall: `{final_metrics['recall'] - baseline_metrics['recall']:+.4f}`; cambio de precisión: `{final_metrics['precision'] - baseline_metrics['precision']:+.4f}`. El mayor recall reduce churners no detectados, pero la menor precisión puede implicar más ofertas de retención innecesarias.
"""
Path('tuning_report.md').write_text(report, encoding='utf-8')
print('Reporte actualizado: tuning_report.md')

,Línea base,Modelo final
accuracy,0.805536,0.740951
precision,0.657233,0.507799
recall,0.558824,0.783422
f1,0.604046,0.616193
f2,0.576075,0.706705
roc_auc,0.841861,0.840967


Matriz baseline [[TN, FP], [FN, TP]]:
 [[926 109]
 [165 209]]
Matriz final [[TN, FP], [FN, TP]]:
 [[751 284]
 [ 81 293]]
Hiperparámetros elegidos: {'classifier__C': 3.39828905, 'classifier__class_weight': 'balanced'}
F2 CV elegido: 0.7228 ± 0.0299
Cambio de recall: +0.2246
Cambio de precisión: -0.1494
Reporte actualizado: tuning_report.md
